# Comparação Unificada — Linhagem `corrected` (`Upgrade/corrected_only/`)

Este notebook refaz a comparação experimental do projeto focando na linhagem **`corrected`** sob o mesmo protocolo completo (`PIGuard Puro`, `+ Laya English Não Calibrado`, `+ Laya English Calibrado`, `TensorTrust`, Latência CUDA e IC 95% via *Paired Bootstrap*):

1. **`augmented_corrected`** (`Corrections`): base corrigida de `75.973` exemplos, **sem MOF**.
2. **`augmented_mof_corrected`** (`Corrections`): base corrigida + **MOF antigo** (`337/237/188` gatilhos).
3. **`train_augmented_corrected_new_mof`** (`Upgrade` Tópico 2): base corrigida + **Novo MOF v2** (`152/229/381` + 4 domínios + HN 80/20).
4. **`augmented_mof_original`** (Âncora `Corrections`): reprodução original antes da correção de dados.
5. **`PIGuard_HF`** (Âncora Oficial): checkpoint público `leolee99/PIGuard` no Hugging Face.

Todos os artefatos existentes em `Upgrade/topic1/`, `topic2/` e `topic3/` são preservados intactos; as novas tabelas, relatórios e gráficos são gravados em **`Upgrade/corrected_only/`**.


## 1. Ambiente, Google Drive, Dependências e Notificações (`ntfy.sh`)


In [ ]:
import os
import sys
import json
import subprocess
import time
import traceback
import urllib.request
from contextlib import contextmanager
from pathlib import Path

NTFY_TOPIC = "piguard_gabriel_treinos"
NTFY_URL = "https://ntfy.sh/"
NTFY_ENABLED = True

def notify_ntfy(title, message, *, priority=3, tags=None, topic=NTFY_TOPIC, timeout=15):
    if not NTFY_ENABLED:
        return
    payload = {
        "topic": topic,
        "title": str(title)[:250],
        "message": str(message)[:3800],
        "priority": int(priority),
    }
    if tags:
        payload["tags"] = list(tags)
    data = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    req = urllib.request.Request(
        NTFY_URL,
        data=data,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            resp.read()
    except Exception as exc:
        print(f"[ntfy] Aviso ao enviar notificação ({title}): {exc}", flush=True)

@contextmanager
def ntfy_cell(cell_name):
    started = time.monotonic()
    notes = []
    try:
        yield notes
    except BaseException as exc:
        elapsed = time.monotonic() - started
        tb_tail = "".join(traceback.format_exception_only(type(exc), exc)).strip()
        msg = f"Célula '{cell_name}' falhou após {elapsed:.1f}s.\nErro: {tb_tail}"
        if notes:
            msg += "\nContexto:\n" + "\n".join(str(n) for n in notes)
        notify_ntfy(
            title=f"[PIGuard Corrected Comparison] Erro: {cell_name}",
            message=msg,
            priority=4,
            tags=["x", "warning"],
        )
        raise
    else:
        elapsed = time.monotonic() - started
        msg = f"Célula '{cell_name}' concluída em {elapsed:.1f}s."
        if notes:
            msg += "\n" + "\n".join(str(n) for n in notes)
        notify_ntfy(
            title=f"[PIGuard Corrected Comparison] Concluída: {cell_name}",
            message=msg,
            priority=3,
            tags=["white_check_mark"],
        )

def run_cmd(command, *, cwd=None):
    proc = subprocess.run(command, cwd=str(cwd) if cwd else None, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Comando falhou ({proc.returncode}): {' '.join(map(str, command))}")

with ntfy_cell("1. Ambiente e Caminhos") as ntfy_notes:
    IS_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
    if IS_COLAB:
        from google.colab import drive, userdata
        drive.mount("/content/drive")

        REPO = Path("/content/Equipe05_PIGuard")
        if not REPO.exists() and Path("/content/RP_PIGuard").exists():
            REPO = Path("/content/RP_PIGuard")
        if not REPO.exists():
            token = None
            try:
                token = (userdata.get("GITHUB_TOKEN") or "").strip() or None
            except Exception:
                token = None
            repo_url = (
                f"https://{token}@github.com/gabrielbelo2007/Equipe05_PIGuard.git"
                if token else "https://github.com/gabrielbelo2007/Equipe05_PIGuard.git"
            )
            run_cmd(["git", "clone", repo_url, str(REPO)])
        elif (REPO / ".git").exists():
            pull = subprocess.run(
                ["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"],
                capture_output=True, text=True,
            )
            if pull.returncode != 0:
                print(f"Aviso git pull: {pull.stderr.strip() or pull.stdout.strip()}")

        run_cmd([
            sys.executable, "-m", "pip", "install", "-q",
            "-r", str(REPO / "Upgrade" / "requirements.txt"),
        ])
    else:
        start = Path(os.environ.get("PIGUARD_REPO_ROOT") or Path.cwd()).expanduser().resolve()
        REPO = next(
            (p for p in (start, *start.parents) if (p / "datasets").is_dir() and (p / "Original").is_dir()),
            None,
        )
        if REPO is None:
            raise FileNotFoundError("Abra o notebook a partir do repositório ou defina PIGUARD_REPO_ROOT.")

    os.chdir(REPO)
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))

    import torch
    import transformers

    from Upgrade import default_corrections_dir, resolve_path
    DRIVE_ROOT = Path("/content/drive/MyDrive/RP_PIGuard_outputs") if IS_COLAB else REPO
    CORRECTIONS_DIR = (DRIVE_ROOT / "Corrections") if IS_COLAB else default_corrections_dir(REPO)
    UPGRADE_OUTPUT = (DRIVE_ROOT / "Upgrade") if IS_COLAB else resolve_path(os.environ.get("PIGUARD_UPGRADE_OUTPUT"), REPO / "Upgrade" / "outputs", repo_root=REPO)
    UPGRADE_OUTPUT.mkdir(parents=True, exist_ok=True)

    SEED = 42
    FPR_BUDGET = 0.01
    N_BOOT = 2000
    FORCE_COMPARISON = False
    FORCE_REVAL_NEW_MOF = False

    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    print({
        "repo": str(REPO),
        "corrections_dir": str(CORRECTIONS_DIR),
        "upgrade_output": str(UPGRADE_OUTPUT),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu": gpu_name,
    })
    ntfy_notes.append(f"gpu={gpu_name} | output={UPGRADE_OUTPUT / 'corrected_only'}")


## 2. Verificação de Pré-Requisitos e Caches do Laya English

Confirma a existência de `Upgrade/step0/calibration_heldout.json`, dos checkpoints de `Corrections/` e dos caches `laya_english_benchmark_logits.json` / `laya_english_heldout_logits.json` em `Upgrade/cache/`. Só instancia o `LayaEnglishEngine` caso algum cache esteja ausente.


In [ ]:
with ntfy_cell("2. Pré-requisitos e Cache Laya") as ntfy_notes:
    import gc
    from Upgrade import load_benchmarks, read_json
    from Upgrade.laya_english_cascade import LayaEnglishEngine
    from Upgrade.topic1_rp_laya_evaluation import extract_or_load_laya_cache

    HELDOUT_PATH = UPGRADE_OUTPUT / "step0" / "calibration_heldout.json"
    if not HELDOUT_PATH.is_file():
        raise FileNotFoundError(f"Conjunto held-out ausente: {HELDOUT_PATH}. Rode a Etapa 0 em upgrade_pipeline.ipynb primeiro.")

    for rel in (
        "augmented_corrected/best_model.pth",
        "augmented_mof_corrected/best_model.pth",
        "augmented_mof_original/best_model.pth",
    ):
        ckpt_file = CORRECTIONS_DIR / rel
        if not ckpt_file.is_file():
            raise FileNotFoundError(f"Checkpoint esperado ausente: {ckpt_file}")

    CACHE_DIR = UPGRADE_OUTPUT / "cache"
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    bench_cache_file = CACHE_DIR / "laya_english_benchmark_logits.json"
    held_cache_file = CACHE_DIR / "laya_english_heldout_logits.json"

    benchmarks = load_benchmarks(REPO / "datasets", include_tensortrust=True)
    heldout_records = read_json(HELDOUT_PATH)

    laya_engine = None
    if not (bench_cache_file.is_file() and held_cache_file.is_file()):
        print("[Cache Laya] Cache não encontrado; carregando LayaEnglishEngine...")
        laya_engine = LayaEnglishEngine(window_max_len=2048, overlap_tokens=512)
    else:
        print("[Cache Laya] Reutilizando logits cacheados no Drive sem carregar ModernBERT-large.")

    bench_laya_cache = extract_or_load_laya_cache(
        {k: texts for k, (texts, _) in benchmarks.items()},
        bench_cache_file,
        laya_engine=laya_engine,
        batch_size=16,
    )
    heldout_laya_cache = extract_or_load_laya_cache(
        {"heldout": [str(r["prompt"]) for r in heldout_records]},
        held_cache_file,
        laya_engine=laya_engine,
        batch_size=16,
    )
    if laya_engine is not None:
        del laya_engine
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    topic2_json = UPGRADE_OUTPUT / "topic2" / "topic2_results.json"
    print({
        "heldout_count": len(heldout_records),
        "laya_bench_keys": list(bench_laya_cache["entries"].keys()),
        "topic2_reusable": topic2_json.is_file(),
    })
    ntfy_notes.append(f"heldout={len(heldout_records)} | topic2_reusable={topic2_json.is_file()}")


## 3. Execução da Comparação Unificada (`augmented_corrected`, `augmented_mof_corrected`, `train_augmented_corrected_new_mof` + Âncoras)

Avalia todos os modelos sob o mesmo protocolo (`Puro`, `+ Laya Não Calibrado`, `+ Laya Calibrado`, `TensorTrust`, Latência CUDA e IC 95% via *Paired Bootstrap*), verifica reprodutibilidade frente a `Corrections/evaluation_raw.json` e salva tabelas e gráficos em `Upgrade/corrected_only/`.


In [ ]:
with ntfy_cell("3. Comparação Unificada Linhagem corrected") as ntfy_notes:
    import pandas as pd
    from IPython.display import Image, Markdown, display
    from Upgrade.corrected_comparison import run_corrected_comparison

    comp_bundle = run_corrected_comparison(
        corrections_dir=CORRECTIONS_DIR,
        heldout_path=UPGRADE_OUTPUT / "step0" / "calibration_heldout.json",
        output_dir=UPGRADE_OUTPUT,
        repo_root=REPO,
        fpr_budget=FPR_BUDGET,
        n_boot=N_BOOT,
        seed=SEED,
        force=FORCE_COMPARISON,
        force_reval_new_mof=FORCE_REVAL_NEW_MOF,
    )

    df = pd.DataFrame(comp_bundle["summary_rows"])
    print("=== Resumo Comparativo (Puro vs + Laya Não Calibrado vs + Laya Calibrado) ===")
    display(df[[
        "display_name", "overdefense", "benign", "malicious", "average",
        "tensor_trust_average", "delta_average_pp", "delta_aug_corr_average_pp",
        "delta_hf_average_pp", "call_rate_pct", "inference_ms", "p95_ms",
        "corrections_ref_max_abs_diff"
    ]].round(3))

    md_file = Path(comp_bundle["markdown_report"])
    if md_file.is_file():
        display(Markdown(md_file.read_text(encoding="utf-8")))

    for cname, cpath in comp_bundle["charts"].items():
        print(f"Gráfico [{cname}]: {cpath}")
        display(Image(filename=cpath))

    for res in comp_bundle["results"]:
        pure_avg = res["modes"]["pure"]["metrics"]["average"]
        cal_avg = res["modes"]["laya_calibrated"]["metrics"]["average"]
        tt_pure = res["modes"]["pure"]["metrics"]["tensor_trust_average"]
        ntfy_notes.append(f"{res['model']}: pure={pure_avg:.2f}% | +laya_cal={cal_avg:.2f}% | TT={tt_pure:.2f}%")
